<a href="https://colab.research.google.com/github/JoyceBodo/chatbot/blob/main/Copy_of_Adaptive_Hybrid_Dense_Sparse_Retrieval_Augmented_Generation_(AHDS_RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Adaptive Hybrid RAG for Financial Statement Analysis
##Notebook 1: Environment Setup and Project Initialization
##Objectives

By the end of this notebook, you will have:

*   Google Drive mounted
*   Project folders created
*   Required libraries installed
*   OpenAI API configured securely
*   Environment tested










#Section 1: Project Overview
# Adaptive Hybrid Dense-Sparse Retrieval-Augmented Generation (AHDS-RAG)

## MSc Research Project

### Title

Adaptive Hybrid Dense-Sparse Retrieval-Augmented Generation Framework for Information Retrieval from Unstructured Financial Statements of Firms Listed on the Nairobi Securities Exchange.

## Notebook 1

Environment Setup and Project Initialisation

This notebook prepares the development environment by:

- Installing libraries
- Mounting Google Drive
- Creating project folders
- Loading API keys securely
- Testing OpenAI connectivity

In [ ]:
# Section 2: Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
##Section 3: Create the Project Folder
import os

PROJECT_PATH = "/content/drive/MyDrive/AHDS_RAG"

folders = [
    "data",
    "data/raw",
    "data/processed",
    "data/chunks",
    "embeddings",
    "vector_db",
    "bm25",
    "models",
    "logs",
    "results",
    "evaluation",
    "notebooks",
    "config"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

print("Project folders created successfully.")

In [ ]:
#Section 4: Install Libraries
!pip install openai
!pip install chromadb
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install sentence-transformers
!pip install pymupdf
!pip install rank-bm25
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install tqdm
!pip install python-dotenv
!pip install tiktoken
!pip install pydantic

In [ ]:
# Section 5: Import Libraries
import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
#Section 6: OpenAI API Key
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

print("API Key Loaded Successfully")

In [ ]:
import json
import os

#Section 8: Save Project Configuration
config = {
    "project_name": "Adaptive Hybrid RAG",
    "vector_database": "ChromaDB",
    "dense_model": "OpenAI Embeddings",
    "sparse_model": "BM25",
    "llm": "GPT-5",
    "country": "Kenya",
    "exchange": "Nairobi Securities Exchange"
}

config_path = os.path.join(PROJECT_PATH, "config", "config.json")

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Configuration saved successfully.")

In [ ]:
#Section 9: Verify the Environment
print("=" * 60)
print("Environment Verification")
print("=" * 60)

print("Project Folder:")
print(PROJECT_PATH)

print("\nContents:")

for item in os.listdir(PROJECT_PATH):
    print("-", item)

print("\nEnvironment is ready!")

##Notebook 2: Collecting and Organizing NSE Annual Reports
#Objectives

By the end of this notebook, you will:

*   Create a structured document repository.
*   Download annual reports (or upload them manually).
*   Organize reports by company and year.
*   Create a metadata catalog (catalog.csv).
*   Validate that all documents are readable.


# Notebook 2: Data Collection and Organisation

## Objectives

This notebook prepares the document corpus for the AHDS-RAG system.

Tasks:
- Organize annual reports
- Build a metadata catalog
- Validate PDF files
- Prepare documents for preprocessing

Output:
- data/raw/
- catalog.csv

In [ ]:
!pip install -q requests
!pip install -q beautifulsoup4
!pip install -q lxml
!pip install -q tqdm

In [ ]:
#Section 2 – Import Libraries
import os
import re
import json
import requests
import shutil
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm

In [ ]:
#Section 3 – Define Project Paths

PROJECT_PATH = "/content/drive/MyDrive/AHDS_RAG"

RAW_DATA = f"{PROJECT_PATH}/data/raw"
META_DATA = f"{PROJECT_PATH}/data/metadata"
LOGS = f"{PROJECT_PATH}/data/logs"

os.makedirs(RAW_DATA, exist_ok=True)
os.makedirs(META_DATA, exist_ok=True)
os.makedirs(LOGS, exist_ok=True)

In [ ]:
BASE_URL = 'https://annualreport.cma.or.ke/business/1/'

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(BASE_URL, headers=headers)

print(response.status_code)

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://annualreport.cma.or.ke/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# Define HEADERS to make it accessible to other functions that might use it
HEADERS = headers

response = requests.get(BASE_URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

business_links = []

for a in soup.find_all("a", href=True):
    href = a.get("href", "")

    if "/business/" in href:
        full_url = urljoin(BASE_URL, href)

        if full_url not in business_links:
            business_links.append(full_url)

print(f"Found {len(business_links)} business sectors:\n")

for link in business_links:
    print(link)

In [ ]:
#Discover Every Company
company_links = []

for business in business_links:

    response = requests.get(business, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    for a in soup.find_all("a", href=True):

        href = a.get("href", "")

        if "/company/" in href:

            full = urljoin(BASE_URL, href)

            if full not in company_links:
                company_links.append(full)

print(f"Found {len(company_links)} companies")

In [ ]:
# inspect one company
test_company = company_links[0]

response = requests.get(test_company, headers=headers)

print(test_company)

print(response.status_code)

print(response.text[:4000])

In [ ]:
#Dataset Builder
import os
import re
import time
import hashlib
import requests
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm

In [ ]:
class DatasetBuilder:

    def __init__(self, base_url):

        self.base_url = base_url

        self.business_links = []

        self.company_links = []

        self.catalog = []

In [ ]:
#Discover Businesses
def discover_businesses(self):

    response = requests.get(self.base_url, headers=HEADERS)

    soup = BeautifulSoup(response.text, "html.parser")

    businesses = []

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/business/" in href:

            link = urljoin(self.base_url, href)

            if link not in businesses:

                businesses.append(link)

    self.business_links = businesses

    print(f"Found {len(businesses)} business sectors")

In [ ]:
#Discover Companies
def discover_companies(self):

    companies = []

    for business in tqdm(self.business_links):

        response = requests.get(business, headers=HEADERS)

        soup = BeautifulSoup(response.text, "html.parser")

        for a in soup.find_all("a", href=True):

            href = a["href"]

            if "/company/" in href:

                link = urljoin(self.base_url, href)

                if link not in companies:

                    companies.append(link)

    self.company_links = companies

    print(f"Found {len(companies)} companies")

In [ ]:
DatasetBuilder.discover_companies = discover_companies

In [ ]:
#Extract Reports
def extract_reports(self):

    reports = []

    for company_url in tqdm(self.company_links):

        response = requests.get(company_url, headers=HEADERS)

        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.find("h4")

        if title:

            text = title.text.strip()

            parts = text.split(">>")

            sector = parts[0].strip()

            company = parts[1].strip()

        else:

            sector = ""

            company = ""

        for a in soup.find_all("a", href=True):

            href = a.get("href", "")

            if href.lower().endswith(".pdf"):

                pdf_url = urljoin(BASE_URL, href)

                filename = os.path.basename(href)

                year_match = re.search(r"\d{4}", filename)

                year = year_match.group() if year_match else ""

                reports.append({

                    "company": company,

                    "sector": sector,

                    "year": year,

                    "filename": filename,

                    "pdf_url": pdf_url

                })

    self.catalog = pd.DataFrame(reports)

    print(f"Found {len(self.catalog)} reports")

In [ ]:
DatasetBuilder.extract_reports = extract_reports

In [ ]:
#Run the Builder
builder = DatasetBuilder(BASE_URL)

# Assign discover_businesses as a method to the DatasetBuilder class
DatasetBuilder.discover_businesses = discover_businesses

builder.discover_businesses()

builder.discover_companies()

builder.extract_reports()

builder.catalog.head()

In [ ]:
catalog_path = os.path.join(META_DATA, "catalog.csv")

builder.catalog.to_csv(catalog_path, index=False)

print(f"Catalog saved to: {catalog_path}")

In [ ]:
#Lets remove duplicates
def extract_reports(self):

    reports = []
    seen = set()

    for company_url in tqdm(self.company_links):

        response = requests.get(company_url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.find("h4")

        if title:
            parts = title.text.strip().split(">>")
            sector = parts[0].strip()
            company = parts[1].strip()
        else:
            sector = ""
            company = ""

        for a in soup.find_all("a", href=True):

            href = a.get("href", "")

            if href.lower().endswith(".pdf"):

                pdf_url = urljoin(BASE_URL, href)

                filename = os.path.basename(href)

                year_match = re.search(r"\d{4}", filename)
                year = year_match.group() if year_match else ""

                key = (company, filename)

                if key not in seen:

                    seen.add(key)

                    reports.append({

                        "company": company,
                        "sector": sector,
                        "year": year,
                        "filename": filename,
                        "pdf_url": pdf_url

                    })

    self.catalog = pd.DataFrame(reports)

    print(f"Unique reports: {len(self.catalog)}")

In [ ]:
builder.extract_reports()

In [ ]:
#Create a download folder
DOWNLOAD_FOLDER = f"{PROJECT_PATH}/data/raw"

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

In [ ]:
#Download a single PDF
from pathlib import Path
import requests

def download_pdf(row):

    company_folder = Path(DOWNLOAD_FOLDER) / row.company
    company_folder.mkdir(parents=True, exist_ok=True)

    destination = company_folder / row.filename

    if destination.exists():
        return "Exists"

    try:

        r = requests.get(row.pdf_url, timeout=60)

        r.raise_for_status()

        with open(destination, "wb") as f:
            f.write(r.content)

        return "Downloaded"

    except Exception as e:

        return str(e)

In [ ]:
#Parallel Download
#This will download multiple reports simultaneously and is much faster than downloading them one at a time.
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

results = []

with ThreadPoolExecutor(max_workers=8) as executor:

    futures = [
        executor.submit(download_pdf, row)
        for _, row in builder.catalog.iterrows()
    ]

    for future in tqdm(futures):

        results.append(future.result())

## Connecting Colab Notebook to GitHub

YouYou can save a copy of your Colab notebook directly to a GitHub repository. Follow these steps:

1.  Go to `File` in the Colab menu bar.
2.  Select `Save a copy in GitHub...`.
3.  A dialog box will appear. You'll need to:
    *   Select the GitHub repository where you want to save the notebook.
    *   Choose whether it's a public or private repository.
    *   Add an optional commit message.
    *   Select the branch (usually `main` or `master`).
4.  If this is your first time connecting Colab to GitHub, you might be prompted to authorize Colab to access your GitHub account. Follow the on-screen instructions.
5.  Click `OK` or `Save`.

Your notebook will then be saved as a `.ipynb` file in the specified GitHub repository.